# AgentFix workshop — Google Colab

In this workshop you will build three missing pieces of a small coding agent and then run it against a real bug-fixing task.

## Important repository rule

Your student repository is always:

`/content/agentfix-workshop`

and it is always based on the GitHub **`main` branch**.

This notebook **never checks out `solutions` or any solution tag in the student checkout**. Solution tags are only read with `git diff` / `git show`, so looking at a solution cannot replace your working files.

The three exercises are:

| Stage | What you implement | File |
|---|---|---|
| 1 | `run_tests` tool + JSON schema | `src/agentfix/tools/tests_tool.py` |
| 2 | Tool dispatch inside the agent loop | `src/agentfix/agent/loop.py` |
| 3 | The agent stop condition | `src/agentfix/agent/loop.py` |


## 0. Before you run anything

In Colab choose **Runtime → Change runtime type → GPU**.

You will edit source files directly in Colab:

1. Click the **folder icon** in the left sidebar.
2. Open `content → agentfix-workshop`.
3. Double-click a `.py` or `.md` file to open it in Colab's editor.
4. Edit and save with **Ctrl+S / Cmd+S**.
5. Return to this notebook and rerun the relevant test cell.

Do **not** use `%%writefile` for the exercises. You are editing the real cloned repository.


In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true


## 1. Workshop configuration

The workshop uses a small local Ollama model so the whole agent can run inside the Colab runtime.


In [ ]:
REPO_URL = "https://github.com/jelenadjuric01/agentfix-workshop.git"
CHECKOUT = "/content/agentfix-workshop"

BASE_MODEL = "qwen2.5-coder:1.5b"
WORKSHOP_MODEL = "agentfix-qwen"
CONTEXT_LENGTH = 16384

print("Repository:", REPO_URL)
print("Student checkout:", CHECKOUT)
print("Student branch: main")
print("Model:", BASE_MODEL, "->", WORKSHOP_MODEL)


Repository: https://github.com/jelenadjuric01/agentfix-workshop.git
Student checkout: /content/agentfix-workshop
Student branch: main
Model: qwen2.5-coder:1.5b -> agentfix-qwen


## 2. Install and start Ollama

Colab may not include `zstd`, which the current Ollama Linux installer needs.

The Ollama server is started in the background and kept available to later cells.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd pciutils
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; fi
!ollama --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.7.0-6) ...
Setting up zstd (1.4.8+dfsg-3bui

In [ ]:
%env OLLAMA_CONTEXT_LENGTH=16384

!if ! curl -fsS http://127.0.0.1:11434/api/version >/dev/null 2>&1; then nohup ollama serve > /tmp/ollama-colab.log 2>&1 & sleep 4; fi
!curl -fsS http://127.0.0.1:11434/api/version


env: OLLAMA_CONTEXT_LENGTH=16384
{"version":"0.32.15"}

## 3. Pull the model and create the workshop model

`MELLUM_MODEL` is the environment variable AgentFix reads for the Ollama model name.


In [ ]:
!ollama pull qwen2.5-coder:1.5b
!printf "FROM qwen2.5-coder:1.5b\nPARAMETER num_ctx 16384\n" > /tmp/Modelfile.agentfix-qwen
!ollama create agentfix-qwen -f /tmp/Modelfile.agentfix-qwen

%env MELLUM_MODEL=agentfix-qwen

!ollama list




env: MELLUM_MODEL=agentfix-qwen
NAME                    ID              SIZE      MODIFIED               
qwen2.5-coder:1.5b      d7372fd82851    986 MB    Less than a second ago    
agentfix-qwen:latest    1329bf6330f5    986 MB    Less than a second ago    


In [ ]:
# Quick model smoke test.
!ollama run agentfix-qwen "Reply with exactly: READY"


READY



# 4. Clone the student repository — clean `main` only

**This is intentionally a clean reset.**

The cell below deletes any previous `/content/agentfix-workshop` directory and clones GitHub `main` again. This guarantees a stale `solutions` checkout or an old Colab editor session cannot contaminate the workshop starting point.

**Do not rerun this cell after you begin editing unless you intentionally want to discard your exercise work.**


In [ ]:
%cd /content

!rm -rf /content/agentfix-workshop
!git clone --branch main https://github.com/jelenadjuric01/agentfix-workshop.git /content/agentfix-workshop

%cd /content/agentfix-workshop

!git fetch --all --tags --prune
!git checkout main
!git reset --hard origin/main
!git clean -fd

!echo "Current branch:"
!git branch --show-current

!echo
!echo "Working tree:"
!git status -sb

!echo
!echo "Local HEAD:"
!git rev-parse HEAD

!echo "origin/main:"
!git rev-parse origin/main


/content
Cloning into '/content/agentfix-workshop'...
remote: Enumerating objects: 637, done.
remote: Counting objects: 100% (637/637), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 637 (delta 340), reused 599 (delta 304), pack-reused 0 (from 0)
Receiving objects: 100% (637/637), 471.56 KiB | 12.74 MiB/s, done.
Resolving deltas: 100% (340/340), done.
/content/agentfix-workshop
Fetching origin
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at 683e5f9 Remove outdated command from Linux installation instructions in README.md.
Current branch:
main

Working tree:
## main...origin/main

Local HEAD:
683e5f9f1b877b7feccec787de68d62b8953d135
origin/main:
683e5f9f1b877b7feccec787de68d62b8953d135


## 5. Verify that `main` contains the exercise stubs

This cell is a guardrail.

It checks that the two exercise source files are byte-for-byte unchanged from `origin/main`, then verifies that all three TODO markers are still present.

Expected:

- Stage 1: **2** TODO markers
- Stage 2: **1** TODO marker
- Stage 3: **1** TODO marker

If these checks pass, you are definitely starting from the exercise version rather than a solution.


In [ ]:
!git diff --exit-code origin/main -- src/agentfix/tools/tests_tool.py src/agentfix/agent/loop.py && echo "OK: student files match origin/main exactly."

!test "$(grep -c 'TODO(stage-1)' src/agentfix/tools/tests_tool.py)" -eq 2 && echo "OK: Stage 1 has 2 TODOs."
!test "$(grep -c 'TODO(stage-2)' src/agentfix/agent/loop.py)" -eq 1 && echo "OK: Stage 2 has 1 TODO."
!test "$(grep -c 'TODO(stage-3)' src/agentfix/agent/loop.py)" -eq 1 && echo "OK: Stage 3 has 1 TODO."

!echo
!grep -n "TODO(stage-1)" src/agentfix/tools/tests_tool.py
!grep -n "TODO(stage-2)" src/agentfix/agent/loop.py
!grep -n "TODO(stage-3)" src/agentfix/agent/loop.py


OK: student files match origin/main exactly.
OK: Stage 1 has 2 TODOs.
OK: Stage 2 has 1 TODO.
OK: Stage 3 has 1 TODO.

28:    # TODO(stage-1): the JSON Schema the model sees. run_tests needs no arguments.
64:        # TODO(stage-1): run the tests via self.backend, store self.last_result,
246:                # TODO(stage-2): dispatch the call through the registry and append the
102:    # TODO(stage-3): the agent is done when the tests actually pass.


## 6. Make accidental GitHub pushes impossible

Everything students do is local to this Colab runtime. The next command also disables the push URL for `origin`, so an accidental `git push` will fail while `git fetch` continues to work.


In [ ]:
!git remote set-url --push origin DISABLED
!git remote -v


origin	https://github.com/jelenadjuric01/agentfix-workshop.git (fetch)
origin	DISABLED (push)


## 7. Install AgentFix in editable mode

Editable mode means changes saved in `src/agentfix/...` are used immediately. You do **not** need to reinstall after every edit.

`agent doctor` should return a [PASS} for everything except ram memory, which is fine.


In [ ]:
!python -m pip install -q -e ".[dev]"
!agentfix doctor


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 21.2 MB/s eta 0:00:00
  Building editable for agentfix (pyproject.toml) ... done
[PASS] python: 3.12.13
[FAIL] ram: 12.7 GB total, 10.

# Start the exercises

Before editing, open the repository in Colab's **Files** sidebar:

`/content/agentfix-workshop`

You can also open the exercise instructions themselves:

- `exercises/stage_1/README.md`
- `exercises/stage_2/README.md`
- `exercises/stage_3/README.md`

You will remain on `main` for the entire workshop.

If you prefer to look at the code from your IDE of choice, feel free to clone the repo and open it yourself: https://github.com/jelenadjuric01/agentfix-workshop


In [ ]:
!git branch --show-current
!git status -sb


main
## main...origin/main


# Stage 1 — Give the agent a tool

### Where to edit

Open:

`src/agentfix/tools/tests_tool.py`

Find the two `TODO(stage-1)` markers.

You need to implement:

1. **`parameters`** — the JSON Schema the model sees.
2. **`run()`** — the function that runs the tests.
Important distinction:

- `ToolResult.ok` means the **tool call itself worked**.
- `self.last_result.passed` means the **project tests passed**.

The full repository guide is in:

`exercises/stage_1/README.md`


In [ ]:
!cat exercises/stage_1/README.md


In [ ]:
!echo "Stage 1 TODO locations:"
!grep -n -A10 -B5 "TODO(stage-1)" src/agentfix/tools/tests_tool.py


### Stage 1 — see the failure first

Before editing, this test is expected to fail. That confirms you really are on the exercise version.

After you edit and save `tests_tool.py`, run the **next** cell until Stage 1 passes.


In [ ]:
!python -m pytest exercises/stage_1 -v || true


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agentfix-workshop
configfile: pyproject.toml
plugins: cov-7.1.0, anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.0
collected 6 items                                                              

exercises/stage_1/test_stage_1.py::test_tool_declares_a_valid_schema PASSED [ 16%]
exercises/stage_1/test_stage_1.py::test_schema_is_exported_to_the_model PASSED [ 33%]
exercises/stage_1/test_stage_1.py::test_running_failing_tests_reports_failure PASSED [ 50%]
exercises/stage_1/test_stage_1.py::test_running_passing_tests_reports_success PASSED [ 66%]
exercises/stage_1/test_stage_1.py::test_the_observation_tells_the_model_what_actually_happened PASSED [ 83%]
exercises/stage_1/test_stage_1.py::test_the_model_chooses_this_tool_when_told_tests_fail PASSED [100%]

============================== 6 pas

### Stage 1 — your test

Edit and save:

`src/agentfix/tools/tests_tool.py`

Then rerun:


In [ ]:
!python -m pytest exercises/stage_1 -v


### Stage 1 — optional solution peek

Only use this if you are stuck.

This compares committed `main` with the `stage-1-solution` tag. It **does not checkout the tag** and **does not change your working files**.


In [ ]:
!git --no-pager diff main stage-1-solution -- src/agentfix/tools/tests_tool.py


diff --git a/src/agentfix/tools/tests_tool.py b/src/agentfix/tools/tests_tool.py
index 4815ffe..bcba46b 100644
--- a/src/agentfix/tools/tests_tool.py
+++ b/src/agentfix/tools/tests_tool.py
@@ -11,7 +11,6 @@ collect it as a test module.
 from __future__ import annotations
 
 from pathlib import Path
-from typing import Any
 
 from agentfix.sandbox.base import ExecResult, ExecutionBackend
 from agentfix.tools.base import ToolResult
@@ -25,8 +24,7 @@ class RunTestsTool:
     # told: nothing hands the agent the failing test output up front, so discovering the
     # failure is part of the task.
     description = "Run the project's test suite and return the result. This is the source of truth."
-    # TODO(stage-1): the JSON Schema the model sees. run_tests needs no arguments.
-    parameters: dict[str, Any] = {}
+    parameters = {"type": "object", "properties": {}}  # takes no arguments
 
     def __init__(
         self,
@@ -58,9 +56,12 @@ class RunTestsTool:
         self.last_result =

# Stage 2 — Close the loop

### Where to edit

Open:

`src/agentfix/agent/loop.py`

Find `TODO(stage-2)` inside `run_agent`.

For each tool call the model made: dispatch it through the registry and append the
result to `messages`.

The full repository guide is in:

`exercises/stage_2/README.md`


In [ ]:
!cat exercises/stage_2/README.md


In [ ]:
!echo "Stage 2 TODO location:"
!grep -n -A16 -B8 "TODO(stage-2)" src/agentfix/agent/loop.py


Stage 2 TODO location:
238-                # Progress: reset the counter and remember this call as the new baseline.
239-                guard_hits = 0
240-                previous_signature = signature
241-                # This is the one place where model output becomes a real effect. Two
242-                # constraints worth knowing before you write it: `registry.dispatch` never
243-                # raises (see tools/base.py — every failure comes back as an observation), and
244-                # the API requires every call the model made to be answered by a message
245-                # carrying its `tool_call_id`.
246:                # TODO(stage-2): dispatch the call through the registry and append the
247-                # resulting tool message to `messages`. Keep the tool_call_id.
248-                tool_started = time.time()
249-                # The one line where model output becomes a real effect. `dispatch` never
250-                # raises — see tools/base.py — so 

### Stage 2 — test

Stage 1 should already pass. Now run Stage 1 + Stage 2 together while you work.


In [ ]:
!python -m pytest exercises/stage_1 exercises/stage_2 -v


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agentfix-workshop
configfile: pyproject.toml
plugins: cov-7.1.0, anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.0
collected 12 items                                                             

exercises/stage_1/test_stage_1.py::test_tool_declares_a_valid_schema PASSED [  8%]
exercises/stage_1/test_stage_1.py::test_schema_is_exported_to_the_model PASSED [ 16%]
exercises/stage_1/test_stage_1.py::test_running_failing_tests_reports_failure PASSED [ 25%]
exercises/stage_1/test_stage_1.py::test_running_passing_tests_reports_success PASSED [ 33%]
exercises/stage_1/test_stage_1.py::test_the_observation_tells_the_model_what_actually_happened PASSED [ 41%]
exercises/stage_1/test_stage_1.py::test_the_model_chooses_this_tool_when_told_tests_fail PASSED [ 50%]
exercises/stage_2/test_stage_2.py::te

### Stage 2 — optional solution peek

This shows **only the change introduced by Stage 2**, by comparing the Stage 1 solution snapshot with the Stage 2 solution snapshot.

Your checkout remains on `main`.


In [ ]:
!git --no-pager diff stage-1-solution stage-2-solution -- src/agentfix/agent/loop.py


# Stage 3 — When is the agent done?

### Where to edit

Stay in:

`src/agentfix/agent/loop.py`

Find `TODO(stage-3)` inside `is_done`.

The tempting answers are both wrong:

- "the model stopped calling tools" — it may have given up, or hallucinated success
- "the model said DONE" — models say DONE about code that does not work

The agent is done when **the tests pass**. Verification by execution, not by assertion.
That is the difference between a demo and something you would let near real code.

The full repository guide is in:

`exercises/stage_3/README.md`


In [ ]:
!cat exercises/stage_3/README.md


In [ ]:
!echo "Stage 3 TODO location:"
!grep -n -A14 -B8 "TODO(stage-3)" src/agentfix/agent/loop.py


### Stage 3 — test

Run all three exercise stages together:


In [ ]:
!python -m pytest exercises/stage_1 exercises/stage_2 exercises/stage_3 -v


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agentfix-workshop
configfile: pyproject.toml
plugins: cov-7.1.0, anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.0
collected 15 items                                                             

exercises/stage_1/test_stage_1.py::test_tool_declares_a_valid_schema PASSED [  6%]
exercises/stage_1/test_stage_1.py::test_schema_is_exported_to_the_model PASSED [ 13%]
exercises/stage_1/test_stage_1.py::test_running_failing_tests_reports_failure PASSED [ 20%]
exercises/stage_1/test_stage_1.py::test_running_passing_tests_reports_success PASSED [ 26%]
exercises/stage_1/test_stage_1.py::test_the_observation_tells_the_model_what_actually_happened PASSED [ 33%]
exercises/stage_1/test_stage_1.py::test_the_model_chooses_this_tool_when_told_tests_fail PASSED [ 40%]
exercises/stage_2/test_stage_2.py::te

### Stage 3 — optional solution peek

This shows only the change introduced by Stage 3.

Again, it reads Git objects only. It never checks out the tag.


In [ ]:
!git --no-pager diff stage-2-solution stage-3-solution -- src/agentfix/agent/loop.py


# Final verification

At this point all three TODOs should be implemented in your working tree.

First run the complete non-model test suite.


In [ ]:
!python -m pytest -q


...............                                                          [100%]
15 passed in 9.57s


In [ ]:
!agentfix doctor


## Run the completed agent

Now use your implementation with the local Ollama model on the first workshop task.


In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose


  step 1 → llm:assistant  [ctx 462 tok, 6.0s]  To find and fix the bug in the test suite, I'll follow these steps:  1. **Run the tests**: First, I'
  step 2 → llm:assistant  [ctx 765 tok, 1.5s]  The latest failure indicates that the `test_function` in the `main.py` file is failing. Here's the f
  step 3 → llm:assistant  [ctx 983 tok, 1.5s]  The latest failure indicates that the `test_function` in the `test_function.py` file is failing. Her
  step 4 → llm:assistant  [ctx 1202 tok, 1.5s]  The latest failure indicates that the `test_function` in the `main.py` file is failing. Here's the f
  step 5 → llm:assistant  [ctx 1419 tok, 1.5s]  The latest failure indicates that the `test_function` in the `main.py` file is failing. Here's the f
  step 6 → llm:assistant  [ctx 1636 tok, 1.6s]  The latest failure indicates that the `test_function` in the `main.py` file is failing. Here's the f
  step 7 → llm:assistant  [ctx 1853 tok, 1.6s]  The latest failure indicates that the `test_function` in the 

## Optional — compare your finished files with the final solution

These commands compare your **current working files** with the final solution tag. They still do not switch branches.

No output means your file matches that solution snapshot exactly.


In [ ]:
!git --no-pager diff stage-3-solution -- src/agentfix/tools/tests_tool.py
!git --no-pager diff stage-3-solution -- src/agentfix/agent/loop.py


## Optional — inspect a full solution file without checking it out

`git show TAG:path` prints a file stored in a tag directly to the notebook output.

Examples:


In [ ]:
# Uncomment only the one you want to inspect.

# !git show stage-1-solution:src/agentfix/tools/tests_tool.py
# !git show stage-2-solution:src/agentfix/agent/loop.py
# !git show stage-3-solution:src/agentfix/agent/loop.py


## Optional — workshop evaluation

Once the implementation is complete, you can run a small model-backed evaluation.


In [ ]:
# Optional: this can take a while because it runs the local model repeatedly.
!agentfix eval --suite workshop --limit 3
